In [1]:
# Semantic Chunking

In [1]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os
# Load environment variables
load_dotenv()
MINIML_MODEL_PATH = os.getenv("MINIML_MODEL_PATH")  ## all-MiniLM-L6-v2-original
MODEL_RERANK = os.getenv("MODEL_RERANK")  ## bge-reranker-base

In [2]:
# Initialize HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name=MINIML_MODEL_PATH)

C:\Users\scientist\AppData\Local\Temp\ipykernel_17340\3398344752.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=MINIML_MODEL_PATH)
D:\NewAgentic\RAG_Terms\venv_chunk\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6445.07it/s]


In [14]:
# Create semantic chunker
splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90  # lower → more sensitive
)

text = """
Python is a programming language. It is widely used in AI.

Machine learning is a subset of AI. It focuses on data-driven models.

The stock market crashed yesterday. Investors are worried.
"""

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk}")


Chunk 1:

Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI. It focuses on data-driven models. The stock market crashed yesterday. Investors are worried. 


In [15]:
# Pre-split into smaller units first (like sentences) so the chunker has more “resolution”
from langchain_text_splitters import CharacterTextSplitter

text = """
Python is a programming language. It is widely used in AI.
Machine learning is a subset of AI. It focuses on data-driven models.
The stock market crashed yesterday. Investors are worried.
"""

# Pre-split into sentences
sentence_splitter = CharacterTextSplitter(separator=". ", chunk_size=1000, chunk_overlap=0)
sentences = sentence_splitter.split_text(text)

# Pass each sentence individually to semantic chunker
final_chunks = []
for sentence in sentences:
    chunks = splitter.split_text(sentence)
    final_chunks.extend(chunks)

for i, chunk in enumerate(final_chunks):
    print(f"\nChunk {i+1}:\n{chunk}")


Chunk 1:
Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI. It focuses on data-driven models. The stock market crashed yesterday. Investors are worried.


In [23]:
text = """
Python is a programming language. It is widely used in AI.
Machine learning is a subset of AI. AI is artificial intelligence. It focuses on AI models.
The stock market crashed yesterday. Investors are worried.
"""

# Initialize embeddings and semantic chunker

# Try different breakpoint threshold types
for threshold_type in ["percentile", "standard_deviation", "interquartile"]:
    print(f"\n--- Using {threshold_type} threshold ---")
    
    splitter = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type=threshold_type,
        breakpoint_threshold_amount=0.3  # Adjust based on your needs
    )
    
    chunks = splitter.split_text(text)
    
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1}:\n{chunk}\n")



--- Using percentile threshold ---
Chunk 1:

Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI.

Chunk 3:
AI is artificial intelligence.

Chunk 4:
It focuses on AI models.

Chunk 5:
The stock market crashed yesterday.

Chunk 6:
Investors are worried.

Chunk 7:



--- Using standard_deviation threshold ---
Chunk 1:

Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 3:
It focuses on AI models. The stock market crashed yesterday. Investors are worried.

Chunk 4:



--- Using interquartile threshold ---
Chunk 1:

Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 3:
It focuses on AI models. The stock market crashed yesterday. Investors are worried.

Chunk 4:


